# Storage Pantry — Instructor Key

Real yenstop snapshot from yen1, 2026-07-10 at 20:56. Every row is one process.  
The script runs `top` in batch mode and captures **all** processes on the node (~3000 rows, not ranked).

This notebook:
1. Loads and visualizes the data
2. Defines what a user, process, CPU, and RAM actually mean from real data
3. Asks and answers questions the class discussion should cover

**Data path:**
- Local Mac: `~/Downloads/yenstop_2026-07-10-20-56-06.csv`
- Yens scratch: `/scratch/shared/gsb-research-computing-ai-skills/data/yenstop_2026-07-10-20-56-06.csv`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

cols = ['timestamp','host','pid','user','pr','ni','virt','res','shr',
        's','cpu_pct','mem_pct','time_plus','command','type']

DATA = os.path.expanduser('~/Downloads/yenstop_2026-07-10-20-56-06.csv')
# DATA = '/scratch/shared/gsb-research-computing-ai-skills/data/yenstop_2026-07-10-20-56-06.csv'

df = pd.read_csv(DATA, header=None, names=cols, on_bad_lines='skip')
print(f'{len(df):,} rows  |  {df["user"].nunique()} unique users  |  host: {df["host"].iloc[0]}')
df.head()

---
## Part 1 — What are we looking at?

Before diving into who's using what, let's ground the vocabulary in the actual data.

### What is a process?

Each row in this dataframe is one **process** — a single running program that the **OS** (operating system — Linux on the Yens; think of it as the kitchen manager that hands out CPU time and memory and keeps programs from stepping on each other) is actively managing. Every process has:

- its own **memory space** — a private block of memory that only it can read or write, so one program can't see or corrupt another's data. This is the `virt` / `res` you see in the columns.
- **CPU usage** — the share of processor time it's using right now (`cpu_pct`).
- a **lifecycle** — it is created, runs, sleeps while waiting, and eventually exits. A process moves between states (`R` running, `S` sleeping) over its life; `time_plus` is how much CPU time it has racked up so far.

Every process gets a **PID** (process ID) from the OS. A process can also **spawn subprocesses** (child processes) — e.g. your **shell** (the command-line program you type commands into — `bash` on the Yens) launches `python`, which may launch a pool of worker processes. Each child gets its own PID and its own memory space, and remembers its parent's PID. That parent → child fan-out is why one user can show up as dozens of processes at once.

In [ ]:
# How many processes does each user have? (user processes only)
proc_counts = df[df['type'] == 'u'].groupby('user').size().sort_values(ascending=False)
print(proc_counts.head(15).to_string())
print(f"\nTotal user processes: {(df['type'] == 'u').sum()}")
print(f"Total system processes: {(df['type'] == 's').sum()}")

### What is a process status?

`s` (status) tells you what the process is doing right now:
- `R` — **running**: actively using CPU
- `S` — **sleeping**: waiting for something (I/O, timer, user input) — but still holding RAM
- `I` — **idle**: kernel-internal threads; safe to ignore for resource analysis

The R/S split is the teaching point. A sleeping process isn't wasting CPU — but it *is* occupying fridge space (RAM).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
counts = df['s'].value_counts()
colors = {'R': '#e74c3c', 'S': '#3498db', 'I': '#bdc3c7'}
labels = {'R': 'R  (running)', 'S': 'S  (sleeping)', 'I': 'I  (idle kernel)'}

bars = ax.bar(range(len(counts)), counts.values,
              color=[colors.get(k, '#95a5a6') for k in counts.index])
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max() * 0.01,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title(f'Process status across all {len(df):,} processes on yen1')
ax.set_xlabel('Status')
ax.set_ylabel('Process count')
ax.set_xticks(range(len(counts)))
# Label from the actual index order — not a hardcoded list — so bars and labels always match
ax.set_xticklabels([labels.get(k, k) for k in counts.index])
ax.set_ylim(0, counts.max() * 1.15)   # headroom so the top count label isn't clipped
plt.tight_layout()
plt.show()
print(f"Only {(df['s']=='R').sum()} processes actively using CPU out of {len(df):,} total.")

### What is CPU usage?

`cpu_pct` is the percentage of **one CPU core** in use by this process. yen1 has **256 cores** (logical CPUs — 128 physical cores × 2 threads; see the threads side quest):
- 100% = one full core completely saturated
- 200% = two cores fully in use (a multi-threaded process)
- 96.9% from one process = ~0.4% of total node capacity

This matters: seeing 96.9% looks alarming at first, but that's ~1 core — the node has 255 other cores.

In [ ]:
# Top users by total CPU (user processes only; drop users with 0% CPU)
top_cpu = (df[df['type'] == 'u']
           .groupby('user')['cpu_pct']
           .sum())
top_cpu = top_cpu[top_cpu > 0].sort_values(ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(top_cpu.index, top_cpu.values, color='#e74c3c', edgecolor='white')
ax.set_xlabel('Total cpu_pct (sum across all user processes)')
ax.set_title('Who is using the most CPU on yen1? (user processes)')
ax.axvline(100, color='gray', linestyle='--', linewidth=0.8, label='1 full core = 100%')
ax.legend(fontsize=9)
for bar, val in zip(bars, top_cpu.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Show the top processes explicitly
print(df.sort_values('cpu_pct', ascending=False)[
    ['user','command','cpu_pct','s','time_plus']
].head(10).to_string(index=False))

### What is memory (RAM)?

Two memory columns, very different meanings:
- `res` — **resident memory**: bytes physically loaded into RAM right now. This is the fridge space **actually in use** — real ingredients on the shelves.
- `virt` — **virtual memory**: address space the process has *reserved*. Like fridge shelves it has **booked ahead** (think groceries pre-ordered) — it may reserve a whole wing but only ever stock a little of it, so `virt` is often far larger than `res`.

Always look at `res` (or `mem_pct`) when asking "is this node running out of RAM?" — `virt` is misleading, because reserved-but-empty shelves don't actually take up any food.

In [ ]:
# Total memory by user (user processes only; drop users with 0% RAM)
top_mem = (df[df['type'] == 'u']
           .groupby('user')['mem_pct']
           .sum())
top_mem = top_mem[top_mem > 0].sort_values(ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(top_mem.index, top_mem.values, color='#3498db', edgecolor='white')
ax.set_xlabel('Total mem_pct (sum across all user processes, % of node RAM)')
ax.set_title('Who is holding the most RAM on yen1? (user processes)')
for bar, val in zip(bars, top_mem.values):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# VIRT vs RES scatter — the gap tells a story
# log-log axes so points don't crush together; ring the claude processes instead of
# stacking overlapping text labels
user_procs = df[(df['type'] == 'u') & (df['virt'] > 0) & (df['res'] > 0)].copy()
user_procs['virt_gb'] = user_procs['virt'] / 1e9
user_procs['res_mb'] = user_procs['res'] / 1e6

fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(user_procs['virt_gb'], user_procs['res_mb'],
                c=user_procs['cpu_pct'], cmap='Reds',
                alpha=0.7, s=30, edgecolors='none')
plt.colorbar(sc, label='cpu_pct (% of one core)')

# highlight claude processes with a ring + single legend entry (no overlapping labels)
claude = user_procs[user_procs['command'] == 'claude']
ax.scatter(claude['virt_gb'], claude['res_mb'],
           facecolors='none', edgecolors='#8e44ad', s=90, linewidths=1.3, label='claude')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Virtual memory (GB, log scale)')
ax.set_ylabel('Resident memory (MB, log scale)')
ax.set_title('VIRT vs RES — user processes on yen1')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

print("claude processes reserve tens of GB of VIRT but keep only hundreds of MB in RES:")
print(claude[['user', 'virt_gb', 'res_mb', 'mem_pct', 'cpu_pct']].round(1).to_string(index=False))

---
## Part 2 — Who is on this node and what are they doing?

In [ ]:
# Command distribution — user processes
cmd_counts = df[df['type'] == 'u']['command'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(8, 4.5))
cmd_counts.sort_values().plot(kind='barh', ax=ax, color='#27ae60', edgecolor='white')
ax.set_xlabel('Process count')
ax.set_title('Most common commands (user processes) — top truncates at 8 chars')
for i, (val, name) in enumerate(zip(cmd_counts.sort_values().values,
                                     cmd_counts.sort_values().index)):
    ax.text(val + 0.2, i, str(val), va='center', fontsize=8)
plt.tight_layout()
plt.show()

print("\nNote: top truncates to 8 chars.")
print("  jupyter+  = jupyterhub-singleuser (notebook kernel)")
print("  remote-+  = remote-viewer or code-server (remote desktop)")
print("  falcon-+  = falcon-sensor (CrowdStrike endpoint agent)")
print("  systemd+  = systemd-journald")

In [ ]:
# Longest-running user processes (string sort on time_plus)
df['_tlen'] = df['time_plus'].str.len()
long_running = (df[df['type'] == 'u']
                .sort_values(['_tlen', 'time_plus'], ascending=False)
                [['user','command','time_plus','cpu_pct','s']]
                .head(10))
df.drop(columns='_tlen', inplace=True)
print(long_running.to_string(index=False))
print()
print("muhua's remote session: ~83 hours (3.5 days) — typical for persistent notebook/remote-desktop sessions.")
print("angikar's python3 job:  ~29 hours of continuous computation.")

---
## Part 3 — Class discussion Q&A

Questions students commonly raise, with answers from the data.

In [ ]:
summary = df[df['type'] == 'u'].groupby('user').agg(
    processes=('pid', 'count'),
    total_cpu=('cpu_pct', 'sum'),
    total_mem=('mem_pct', 'sum'),
    running=('s', lambda x: (x == 'R').sum()),
    tools=('command', lambda x: ', '.join(sorted(x.unique())))
).sort_values('total_cpu', ascending=False)
summary

**Q: How many users are on yen1 right now?**  
A: 50 unique user accounts have at least one process. But only 2 are actively computing (angikar, lenardst).

**Q: With two users at 96.9% CPU each — is the node overwhelmed?**  
A: No. 96.9% = one full core. yen1 has 256. They're each using 1/256 of total node capacity. The node has enormous headroom.

**Q: Then why do shared nodes feel slow?**  
A: Usually it's RAM, not CPU. yen1 has ~1 TB of RAM, and `mem_pct` is each process's share of that. Check `lenardst` — 2.4% of node RAM ≈ **24 GB** (confirm it: `res` = 24,400,000,000 bytes ≈ 24.4 GB). One or two users at that level is fine, but if many researchers each hold tens of GB, the fridge fills up even while the stoves (CPUs) sit idle.

**Q: What's the monitoring script doing in its own data?**  
A: `topdump` (the yenstop script) shows up at 46.9% CPU — it's running `top` itself, which counts as CPU usage. Data collection has a footprint. This is a nice meta-observation: every instrument changes what it measures.

**Q: Is muhua's process a SLURM job?**  
A: No — `remote-+` is a remote desktop or code-server session, not a batch job. It's been alive 3.5 days because the user left it running and the Yens allow it. Batch jobs get killed when they exceed their time limit; interactive sessions don't unless an admin intervenes.

**Q: Why do claude processes show 70GB VIRT but only 450MB RES?**  
A: Claude Code memory-maps its model context and project files — the OS reserves address space up front but only loads pages into RAM when they're actually accessed. The 70GB is like reserving an entire wing of the fridge; the 450MB is what's actually inside it.